In [ ]:
from pathlib import Path
import shutil

def organize_files(source_dir_path):
    # 將字串路徑轉換為 Path 物件
    source_dir = Path(source_dir_path)
    uncategorized_dir = source_dir / "未分類"

    # 1. 建立檔案清單 (過濾掉資料夾以及 Mac 系統的 . 開頭隱藏檔)
    file_list = [
        f for f in source_dir.iterdir() 
        if f.is_file() and not f.name.startswith('.')
    ]

    print(f"共找到 {len(file_list)} 個待處理檔案。清單如下：")
    
    # 逐行印出檔名，方便閱讀
    for file in file_list:
        print(file.name)

    for file_path in file_list:
        filename = file_path.name

        # 2. 處理沒有底線的例外檔案
        if '_' not in filename:
            uncategorized_dir.mkdir(exist_ok=True)
            target_dir = uncategorized_dir
        else:
            # 3. 抓出 _ 前面的那串值，並設定目標資料夾路徑
            prefix = filename.split('_')[0]
            target_dir = source_dir / prefix
            # 如果資料夾不存在，就建立一個
            target_dir.mkdir(exist_ok=True)

        target_file_path = target_dir / filename

        # 4. 處理直接覆蓋的邏輯
        if target_file_path.exists():
            target_file_path.unlink()  # 先刪除已存在的舊檔案

        # 5. 移動檔案
        shutil.move(str(file_path), str(target_dir))
        print(f"已移動: {filename} -> {target_dir.name}/")

    print("檔案整理完成！")

# === 執行區塊 ===
# 請將下方引號內的路徑，替換成你 Mac 上的實際資料夾路徑
# 例如: '/Users/你的使用者名稱/Documents/earthquake_data'
target_folder = r'D:\filtered_by_Hualien_events_extracted_waveforms'
organize_files(target_folder)








共找到 36974 個待處理檔案。清單如下：
1309051838_TWY_HL_SMT_10.npy
1612171955_NTS_HL_BH_10.npy
1203281329_NWF_HL_SMT_10.npy
1502040814_HWA_HL_SMT_10.npy
1203051246_HWA_HL_BH_10.npy
1603181811_ETLH_HL_BH_10.npy
1512021159_EHP_HL_SMT_10.npy
1604300702_FULB_HL_BB_10.npy
1403250320_TWF1_HL_SMT_10.npy
1411142211_EYUL_HL_BH_10.npy
1206200854_TWD_HL_SMT_10.npy
1206150950_NWF_HL_SMT_10.npy
1303120410_EHY_HL_SMT_10.npy
1606092115_NWL_HL_BH_10.npy
1603031028_EHY_HL_SMT_10.npy
1410190950_EYUL_HL_BH_10.npy
1605222107_TWF1_HL_SMT_10.npy
1510151857_TAP_HL_SMT_10.npy
1306021016_NWL_HL_BH_10.npy
1211101320_TWY_HL_SMT_10.npy
1602180112_EGC_HL_SMT_10.npy
1612050328_TAP_HL_SMT_10.npy
1412310754_TWD_HL_SMT_10.npy
1403271435_NWF_HL_SMT_10.npy
1303131006_TWA_HL_SMT_10.npy
1206092100_EGFH_HL_BH_10.npy
1403240712_TWY_HL_SMT_10.npy
1605062259_TWB1_HL_BB_10.npy
1511210409_FULB_HL_BB_10.npy
1604030151_TIPB_HL_BB_10.npy
1604260145_HWA_HL_SMT_10.npy
1405310436_NTS_HL_BH_10.npy
1201100740_TWS1_HL_SMT_10.npy
1503271909_EHY_HL_SMT_

In [3]:
from pathlib import Path

def get_all_file_names_recursive(source_dir_path):
    source_dir = Path(source_dir_path)
    
    # 使用 rglob('*') 來遞迴搜尋該目錄與所有子目錄下的檔案
    # 同樣保留了過濾資料夾與 Mac 隱藏檔的判斷
    file_names_list = [
        f.name for f in source_dir.rglob('*') 
        if f.is_file() and not f.name.startswith('.')
    ]
    
    return file_names_list

# === 執行區塊 ===
target_folder = '/Users/lin/碩班/地震模型/目前模型/processdata/extracted_waveforms'

# 執行函式，並將結果存進 all_recursive_files 這個變數中
all_recursive_files = get_all_file_names_recursive(target_folder)

# 檢查變數內容
print(f"變數建立成功！包含子資料夾，裡面總共有 {len(all_recursive_files)} 個檔案。")

# 印出前 5 個檔名來抽樣檢查
print("前 5 個檔案名稱如下：")
for name in all_recursive_files[:5]:
    print(name)


import pandas as pd
from pathlib import Path

def remove_traces_from_csv(npy_files_list, csv_file_path, output_csv_path=None):
    # 1. 建立黑名單：去掉 .npy 拿乾淨的 trace_name
    # 使用 pathlib 的 .stem 屬性可以非常方便地直接取得「不含副檔名的主檔名」
    # 這裡轉成 set (集合) 是為了加快後續 pandas 比對搜尋的速度
    traces_to_remove = set(Path(f).stem for f in npy_files_list)
    print(f"已建立黑名單，共有 {len(traces_to_remove)} 筆 trace_name 準備刪除。")

    # 2. 讀取 CSV 檔案
    print("正在讀取 CSV 檔案...")
    try:
        df = pd.read_csv(csv_file_path)
    except FileNotFoundError:
        print(f"找不到 CSV 檔案：{csv_file_path}")
        return

    # 檢查 CSV 是否真的有 'trace_name' 這個欄位
    if 'trace_name' not in df.columns:
        print("錯誤：CSV 檔案中找不到 'trace_name' 欄位！")
        return

    original_count = len(df)

    # 3. 比對與篩選 (過濾掉在黑名單內的資料)
    # ~ 符號代表「反向」，意思是：保留 df['trace_name'] "不包含在" (isin) 黑名單裡面的資料
    df_filtered = df[~df['trace_name'].isin(traces_to_remove)]
    
    new_count = len(df_filtered)
    deleted_count = original_count - new_count
    
    print(f"比對完成！原本有 {original_count} 筆資料，刪除了 {deleted_count} 筆。")
    print(f"過濾後剩下 {new_count} 筆資料。")

    # 4. 存檔
    # 如果沒有特別指定輸出路徑，預設會在原檔名後面加上 _cleaned
    if output_csv_path is None:
        csv_path_obj = Path(csv_file_path)
        output_csv_path = csv_path_obj.parent / f"{csv_path_obj.stem}_cleaned.csv"
    
    df_filtered.to_csv(output_csv_path, index=False)
    print(f"已將更新後的資料儲存至：{output_csv_path}")


# === 執行區塊 ===


# 請替換成你實際的 CSV 檔案路徑
target_csv = '/Users/lin/碩班/地震模型/目前模型/source/unfinish.csv'

# 執行過濾程式
remove_traces_from_csv(all_recursive_files, target_csv)

變數建立成功！包含子資料夾，裡面總共有 36974 個檔案。
前 5 個檔案名稱如下：
1209131416_EGFH_HL_BH_10.npy
1209131416_TWS1_HL_SMT_10.npy
1209131416_ESL_HL_SMT_10.npy
1209131416_TWD_HL_SMT_10.npy
1209131416_NWL_HL_BH_10.npy
已建立黑名單，共有 36974 筆 trace_name 準備刪除。
正在讀取 CSV 檔案...
比對完成！原本有 98532 筆資料，刪除了 36974 筆。
過濾後剩下 61558 筆資料。
已將更新後的資料儲存至：/Users/lin/碩班/地震模型/目前模型/source/unfinish_cleaned.csv


In [4]:
import pandas as pd
from pathlib import Path

def remove_traces_from_csv(npy_files_list, csv_file_path, output_csv_path=None):
    # 1. 建立黑名單：去掉 .npy 拿乾淨的 trace_name
    # 使用 pathlib 的 .stem 屬性可以非常方便地直接取得「不含副檔名的主檔名」
    # 這裡轉成 set (集合) 是為了加快後續 pandas 比對搜尋的速度
    traces_to_remove = set(Path(f).stem for f in npy_files_list)
    print(f"已建立黑名單，共有 {len(traces_to_remove)} 筆 trace_name 準備刪除。")

    # 2. 讀取 CSV 檔案
    print("正在讀取 CSV 檔案...")
    try:
        df = pd.read_csv(csv_file_path)
    except FileNotFoundError:
        print(f"找不到 CSV 檔案：{csv_file_path}")
        return

    # 檢查 CSV 是否真的有 'trace_name' 這個欄位
    if 'trace_name' not in df.columns:
        print("錯誤：CSV 檔案中找不到 'trace_name' 欄位！")
        return

    original_count = len(df)

    # 3. 比對與篩選 (過濾掉在黑名單內的資料)
    # ~ 符號代表「反向」，意思是：保留 df['trace_name'] "不包含在" (isin) 黑名單裡面的資料
    df_filtered = df[~df['trace_name'].isin(traces_to_remove)]
    
    new_count = len(df_filtered)
    deleted_count = original_count - new_count
    
    print(f"比對完成！原本有 {original_count} 筆資料，刪除了 {deleted_count} 筆。")
    print(f"過濾後剩下 {new_count} 筆資料。")

    # 4. 存檔
    # 如果沒有特別指定輸出路徑，預設會在原檔名後面加上 _cleaned
    if output_csv_path is None:
        csv_path_obj = Path(csv_file_path)
        output_csv_path = csv_path_obj.parent / f"{csv_path_obj.stem}_cleaned.csv"
    
    df_filtered.to_csv(output_csv_path, index=False)
    print(f"已將更新後的資料儲存至：{output_csv_path}")


# === 執行區塊 ===
# 這裡假設 all_my_files 是你剛剛用前面程式碼抓出來的檔案清單變數
# 為了測試，我先手動建幾個你剛剛提供的範例檔名
all_my_files = [
    '1209131416_EGFH_HL_BH_10.npy',
    '1209131416_TWS1_HL_SMT_10.npy',
    '1209131416_ESL_HL_SMT_10.npy',
    '1209131416_TWD_HL_SMT_10.npy',
    '1209131416_NWL_HL_BH_10.npy'
]

# 請替換成你實際的 CSV 檔案路徑
target_csv = '/請替換成/你的/目標CSV路徑/metadata.csv'

# 執行過濾程式
remove_traces_from_csv(all_my_files, target_csv)

已建立黑名單，共有 5 筆 trace_name 準備刪除。
正在讀取 CSV 檔案...
找不到 CSV 檔案：/請替換成/你的/目標CSV路徑/metadata.csv


In [5]:
from pathlib import Path

def get_all_file_names_recursive(source_dir_path):
    source_dir = Path(source_dir_path)
    
    # 使用 rglob('*') 來遞迴搜尋該目錄與所有子目錄下的檔案
    # 同樣保留了過濾資料夾與 Mac 隱藏檔的判斷
    file_names_list = [
        f.name for f in source_dir.rglob('*') 
        if f.is_file() and not f.name.startswith('.')
    ]
    
    return file_names_list

# === 執行區塊 ===
target_folder = '/Users/lin/碩班/地震模型/目前模型/processdata/extracted_waveforms'

# 執行函式，並將結果存進 all_recursive_files 這個變數中
all_recursive_files = get_all_file_names_recursive(target_folder)

# 檢查變數內容
print(f"變數建立成功！包含子資料夾，裡面總共有 {len(all_recursive_files)} 個檔案。")


變數建立成功！包含子資料夾，裡面總共有 36974 個檔案。
